In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

In [2]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

In [3]:
train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

100%|██████████| 9.91M/9.91M [00:32<00:00, 306kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 102kB/s]
100%|██████████| 1.65M/1.65M [00:17<00:00, 91.9kB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.23MB/s]


In [4]:
train_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_data, batch_size=64, shuffle=False)

In [5]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(784, 128)   # Hidden layer 1
        self.fc2 = nn.Linear(128, 64)    # Hidden layer 2
        self.fc3 = nn.Linear(64, 10)     # Output layer

    def forward(self, x):
        x = x.view(-1, 784)              # Flatten image
        x = torch.relu(self.fc1(x))      # Activation 1
        x = torch.relu(self.fc2(x))      # Activation 2
        x = self.fc3(x)                  # Output logits
        return x


In [6]:
model = MLP()
criterion = nn.CrossEntropyLoss()        # Cross‑entropy loss
optimizer = optim.Adam(model.parameters(), lr=0.001)


***Training***

In [7]:
epochs = 5
for epoch in range(epochs):
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")


Epoch 1/5, Loss: 0.1783
Epoch 2/5, Loss: 0.1696
Epoch 3/5, Loss: 0.2154
Epoch 4/5, Loss: 0.0103
Epoch 5/5, Loss: 0.1258


***evaluation***

In [8]:
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy: {100 * correct / total:.2f}%")


Accuracy: 96.74%
